# ValoStats — Notebook 2: análisis predictivo de jugador reciente

## 1. Tareas de aprendizaje

El proyecto se organiza en cuatro tareas principales:

| Tarea | Entrada | Variable objetivo | Salida esperada |
|---|---|---|---|
| Clasificación de rendimiento | Métricas por partida como ACS, K/D, ADR, KAST, DDA, HS%, resultado y rondas jugadas | Nivel de rendimiento | Bajo, Medio, Alto o Destacado |
| Clasificación de estilo | Variables como agresividad, precisión, impacto, soporte, eficiencia, entry power y consistencia | Tipo de jugador | Alto impacto, Apoyo táctico u Ofensivo consistente |
| Análisis de tendencia | Comparación entre partidas antiguas y recientes dentro de las últimas 20 partidas | Tendencia reciente | Riesgo de bajar, Estable, Progreso positivo o Subida probable |
| Jugadores similares | Vector resumen del jugador y base de referencia | Similitud competitiva | Referentes similares en lobbies de rango parecido |

## 3. Métricas competitivas utilizadas

Para que el análisis no se base solo en estadísticas generales, ValoStats incorpora métricas competitivas usadas comúnmente en Valorant.

| Métrica | Estado en el proyecto | Uso dentro del análisis |
|---|---|---|
| ACS | Implementada | Impacto general por ronda. Se usa en rendimiento, tendencia y comparación. |
| ADR | Implementada | Daño promedio por ronda. Mide impacto ofensivo más allá de kills. |
| K/D | Implementada | Eficiencia en duelos. |
| KAST | Implementada | Participación y consistencia por ronda. |
| Headshot % | Implementada | Precisión mecánica. |
| First kills | Implementada | Iniciativa e impacto en duelos iniciales. |
| First deaths | Implementada | Riesgo en duelos iniciales. |
| Entry success | Implementada | Se calcula con first kills y first deaths. |
| DDA | Implementada | Diferencia de daño. |
| TRS / Tracker Score | Implementada | Puntaje general de Tracker.gg usado como apoyo. |
| Multi kills | Implementada | Impacto múltiple en rondas. |
| Clutch rate | No implementada directamente | No se calcula como tasa real porque no se obtienen de forma consistente intentos de clutch y clutches ganados. |
| Utility impact | Aproximada | Se aproxima mediante asistencias, soporte, KAST y participación por ronda. |

Esta decisión permite trabajar con métricas reales disponibles y declarar como limitación aquello que no se puede extraer de forma consistente.

## 4. Configuración inicial

Este notebook puede ejecutarse de dos formas:

- Modo rápido: usa el archivo `data/recent_matches.csv` ya generado.
- Modo completo: ejecuta el scraper.

Para una presentación se recomienda usar el modo rápido, porque el scraping completo puede tardar varios minutos.

In [2]:
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        has_src = (candidate / "src").exists()
        has_data = (candidate / "data").exists()

        if has_src and has_data:
            return candidate

    if (current.parent / "src").exists():
        return current.parent

    return current

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

print("Raíz del proyecto:", PROJECT_ROOT)

Raíz del proyecto: C:\Users\jean_\OneDrive\Escritorio\Mineria de datos\Proyecto\Valostats


## 5. Ejecución del pipeline completo

El script `src/run_full_analysis.py` centraliza todo el flujo.

Si `RUN_SCRAPER = False`, se reutiliza `data/recent_matches.csv`.

Si `RUN_SCRAPER = True`, el sistema abre Tracker.gg, extrae las últimas partidas competitivas y luego ejecuta las predicciones.

In [3]:
RIOT_ID = "PoloGB#LAS"

# Cambiar a True solo si se quiere ejecutar el scraper real.
RUN_SCRAPER = True

# Cambiar a True cuando se actualice data/rank_reference_matches.csv y se quiera reconstruir data/rank_reference_profiles.csv.
REFRESH_REFERENCE = False

command = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "run_full_analysis.py"),
    RIOT_ID,
]

if not RUN_SCRAPER:
    command.append("--skip-scraper")

if REFRESH_REFERENCE:
    command.append("--refresh-reference")

print("Comando a ejecutar:")
print(" ".join(command))

Comando a ejecutar:
c:\Users\jean_\OneDrive\Escritorio\Mineria de datos\Proyecto\Valostats\.venv\Scripts\python.exe C:\Users\jean_\OneDrive\Escritorio\Mineria de datos\Proyecto\Valostats\src\run_full_analysis.py PoloGB#LAS


In [4]:
run_pipeline = True

if run_pipeline:
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("El pipeline falló. Revisar salida anterior.")
else:
    print("Pipeline no ejecutado desde el notebook. Se usará el JSON final existente.")


🚀 Iniciando análisis completo de ValoStats
🎮 Jugador objetivo: PoloGB#LAS
📁 Proyecto: C:\Users\jean_\OneDrive\Escritorio\Mineria de datos\Proyecto\Valostats

▶ Ejecutando paso: Scraper de partidas recientes
Comando:
c:\Users\jean_\OneDrive\Escritorio\Mineria de datos\Proyecto\Valostats\.venv\Scripts\python.exe C:\Users\jean_\OneDrive\Escritorio\Mineria de datos\Proyecto\Valostats\data\tracker\scraper_valorant.py PoloGB#LAS

🚀 Abriendo Chrome en modo debugging...
   C:\Program Files\Google\Chrome\Application\chrome.exe
⏳ Esperando 10 segundos para que Chrome inicie...
✅ Chrome debugging iniciado correctamente
🔌 Conectando al Chrome real abierto en modo debugging...
✅ Conectado al Chrome real
🌐 Página objetivo: https://tracker.gg/valorant/profile/riot/PoloGB%23LAS/matches?playlist=competitive&platform=pc
🌐 Abriendo página en el Chrome real...
⏳ Esperando carga del perfil (1/3)...
✅ Partidas detectadas automáticamente: 20
✅ Partidas visibles encontradas: 20
📌 Se intentarán abrir 20 parti

## 6. Carga del análisis final

El archivo `final_player_analysis.json` unifica todas las salidas del sistema:

- Predicción de rendimiento.
- Predicción de estilo.
- Tendencia temporal.
- Jugadores similares.
- Recomendaciones.
- Predicción por partida.
- Metodología.

In [5]:
final_path = PROJECT_ROOT / "outputs" / "recent_predictions" / "final_player_analysis.json"

if not final_path.exists():
    raise FileNotFoundError(
        f"No existe {final_path}. Ejecuta primero el pipeline o genera el análisis desde la página."
    )

with open(final_path, "r", encoding="utf-8") as file:
    analysis = json.load(file)

analysis.keys()

dict_keys(['player', 'summary', 'prediction_summary', 'temporal_evolution', 'rank_context', 'similar_group_summary', 'gap_analysis', 'similar_players', 'recommendations', 'matches', 'methodology', 'source_files'])

## 7. Resumen del jugador analizado

In [6]:
player = analysis["player"]
summary = analysis["summary"]
prediction = analysis["prediction_summary"]

player_summary = pd.DataFrame([
    {
        "Riot ID": player.get("riot_id"),
        "Rango actual": player.get("current_rank"),
        "Partidas analizadas": summary.get("matches_analyzed"),
        "Winrate": summary.get("winrate"),
        "K/D reciente": summary.get("recent_kd"),
        "ACS reciente": summary.get("recent_acs"),
        "ADR reciente": summary.get("recent_adr"),
        "KAST reciente": summary.get("recent_kast"),
        "Rendimiento global": prediction.get("performance_level"),
        "Estilo principal": prediction.get("main_style"),
        "Estilo secundario": prediction.get("secondary_style"),
        "Tendencia": prediction.get("trend_status"),
    }
])

player_summary

,Riot ID,Rango actual,Partidas analizadas,Winrate,K/D reciente,ACS reciente,ADR reciente,KAST reciente,Rendimiento global,Estilo principal,Estilo secundario,Tendencia
0,PoloGB#LAS,Platinum 2,20,55.0,1.242,222.85,150.44,75.65,Alto,Apoyo táctico,Alto impacto,Subida probable


## 8. Predicción de rendimiento

El rendimiento se clasifica por partida y también a nivel global.

Las clases usadas son:

- Bajo
- Medio
- Alto
- Destacado

In [7]:
performance_distribution = pd.DataFrame(
    list(prediction["performance_distribution"].items()),
    columns=["Rendimiento", "Cantidad de partidas"]
)

performance_distribution

,Rendimiento,Cantidad de partidas
0,Bajo,6
1,Medio,3
2,Alto,7
3,Destacado,4


In [8]:
print("Estado competitivo:")
print(prediction["competitive_status"])

Estado competitivo:
Rendimiento acorde o levemente por encima de Platinum 2. Presenta irregularidad por varias partidas de bajo rendimiento.


## 9. Predicción de estilo de juego

El estilo de juego se predice usando los perfiles construidos en el Notebook 1.

Las clases usadas son:

- Alto impacto
- Apoyo táctico
- Ofensivo consistente

In [9]:
style_distribution = pd.DataFrame(
    list(prediction["style_distribution"].items()),
    columns=["Estilo", "Cantidad de partidas"]
)

style_distribution

,Estilo,Cantidad de partidas
0,Alto impacto,8
1,Apoyo táctico,9
2,Ofensivo consistente,3


## 10. Evolución temporal del jugador

Para incorporar evolución temporal, el sistema no analiza las últimas 20 partidas como un solo bloque estático.

En cambio, divide las partidas en dos tramos:

- Partidas 11 a 20: tramo anterior.
- Partidas 1 a 10: tramo reciente.

Esto permite observar progreso, caída o estabilidad.

In [10]:
temporal = analysis.get("temporal_evolution")

if temporal is None:
    # Fallback por si el JSON fue generado antes de agregar temporal_evolution.
    trend_path = PROJECT_ROOT / "outputs" / "recent_predictions" / "trend_predictions.json"
    with open(trend_path, "r", encoding="utf-8") as file:
        trend_payload = json.load(file)
    temporal = trend_payload["global_trend_prediction"]

previous_half = temporal["previous_half"]
recent_half = temporal["recent_half"]
deltas = temporal["deltas"]

temporal_table = pd.DataFrame([
    {
        "Métrica": "Winrate",
        "Partidas 11-20": previous_half["winrate"],
        "Partidas 1-10": recent_half["winrate"],
        "Cambio": deltas["winrate"],
    },
    {
        "Métrica": "ACS promedio",
        "Partidas 11-20": previous_half["avg_acs"],
        "Partidas 1-10": recent_half["avg_acs"],
        "Cambio": deltas["avg_acs"],
    },
    {
        "Métrica": "K/D promedio",
        "Partidas 11-20": previous_half["avg_kd"],
        "Partidas 1-10": recent_half["avg_kd"],
        "Cambio": deltas["avg_kd"],
    },
    {
        "Métrica": "KAST promedio",
        "Partidas 11-20": previous_half["avg_kast"],
        "Partidas 1-10": recent_half["avg_kast"],
        "Cambio": deltas["avg_kast"],
    },
    {
        "Métrica": "Estilo principal",
        "Partidas 11-20": previous_half["main_style"],
        "Partidas 1-10": recent_half["main_style"],
        "Cambio": f"{previous_half['main_style']} -> {recent_half['main_style']}",
    },
])

temporal_table

,Métrica,Partidas 11-20,Partidas 1-10,Cambio
0,Winrate,30.0,80.0,50.0
1,ACS promedio,191.8,253.9,62.1
2,K/D promedio,1.0,1.66,0.66
3,KAST promedio,70.6,80.7,10.1
4,Estilo principal,Apoyo táctico,Alto impacto,Apoyo táctico -> Alto impacto


In [11]:
print("Tendencia global:")
print(temporal["trend_status"])
print()
print("Explicación:")
print(temporal["trend_explanation"])

Tendencia global:
Subida probable

Explicación:
Tendencia global subida probable: winrate subió 50.0 puntos, ACS promedio subió 62.1, K/D promedio subió 0.66, KAST subió 10.1 puntos, cambio de estilo principal: Apoyo táctico → Alto impacto.


## 11. Sistema de recomendación con jugadores similares

Para formalizar el sistema de recomendación, se implementó una búsqueda de jugadores similares usando Nearest Neighbors y similitud coseno.

El procedimiento es:

1. El jugador analizado se resume como un vector de métricas recientes.
2. Se calcula su media de rango de lobby.
3. Se filtra la base de referencia para usar jugadores de lobbies similares.
4. Se aplica Nearest Neighbors con similitud coseno.
5. Se calculan brechas contra el grupo similar.
6. Se generan recomendaciones.

Esto permite que las recomendaciones no dependan solo de reglas fijas, sino de una comparación vectorial con jugadores de contexto competitivo parecido.

In [12]:
rank_context = analysis["rank_context"]
similar_summary = analysis["similar_group_summary"]
gap_analysis = analysis["gap_analysis"]

pd.DataFrame([
    {
        "Media de lobby": rank_context.get("target_avg_team_rank_nearest"),
        "Grupo de lobby": rank_context.get("target_avg_team_rank_group"),
        "Filtro usado": rank_context.get("filter_info", {}).get("filter_type"),
        "Jugadores comparados": similar_summary.get("players_compared"),
        "Winrate grupo similar": similar_summary.get("avg_winrate"),
        "K/D grupo similar": similar_summary.get("avg_kd"),
        "ACS grupo similar": similar_summary.get("avg_acs"),
    }
])

,Media de lobby,Grupo de lobby,Filtro usado,Jugadores comparados,Winrate grupo similar,K/D grupo similar,ACS grupo similar
0,Gold 1,Gold,same_rank_group,7,48.57,0.995,209.51


In [13]:
similar_players = pd.DataFrame(analysis["similar_players"])

similar_players[
    [
        "rank",
        "reference_riot_id",
        "current_rank_mode",
        "avg_team_rank_nearest",
        "winrate",
        "recent_kd",
        "recent_acs",
        "recent_adr",
        "recent_kast",
        "main_agent",
    ]
].head(10)

,rank,reference_riot_id,current_rank_mode,avg_team_rank_nearest,winrate,recent_kd,recent_acs,recent_adr,recent_kast,main_agent
0,1,LOCO VITOKO#LAS,Platinum 2,Gold 1,60.0,0.963,228.85,144.42,69.65,Gekko
1,2,Valuu7K#go7k,Gold 2,Gold 1,50.0,1.083,204.95,130.94,68.45,Sage
2,3,メMonKlawメ#8422,Platinum 1,Gold 1,65.0,1.077,200.45,126.00,68.95,Killjoy
3,4,KCHACHILENOS#9051,Platinum 3,Gold 2,30.0,0.968,220.00,138.50,70.65,Phoenix
4,5,TutoEtern0#6778,Platinum 1,Gold 2,45.0,1.026,217.95,129.31,64.00,Chamber
5,6,Roquefelerrr#OMFG,Platinum 1,Gold 1,50.0,0.926,196.05,129.03,70.05,Jett
6,7,1D3R#edge,Gold 2,Gold 1,40.0,0.919,198.30,121.30,66.85,Sova


In [14]:
gap_table = pd.DataFrame(
    list(gap_analysis.items()),
    columns=["Brecha", "Valor"]
)

gap_table

,Brecha,Valor
0,winrate_gap,6.430
1,kd_gap,0.247
2,acs_gap,13.340
3,trs_gap,97.430
4,adr_gap,19.080
5,dda_gap,19.980
6,hs_gap,-3.260
7,kast_gap,7.280
8,entry_success_gap,0.760


## 12. Recomendaciones generadas

Las recomendaciones se construyen a partir de:

- Rendimiento global.
- Estilo principal y secundario.
- Tendencia reciente.
- Brechas frente al grupo de jugadores similares.

In [15]:
for index, recommendation in enumerate(analysis["recommendations"], start=1):
    print(f"{index}. {recommendation}")

1. Tu K/D supera al grupo similar; estás ganando más duelos que jugadores con lobbies parecidos.
2. Tu KAST está sobre el grupo similar; participas bien en las rondas incluso cuando no siempre consigues kills.
3. Tu estilo principal es Apoyo táctico; mantén utilidad y asistencias, pero busca convertir más rondas en impacto directo.
4. También aparece como estilo secundario Alto impacto, por lo que tu perfil reciente no es único y mezcla más de una forma de juego.
5. La tendencia reciente indica subida probable si mantienes el nivel mostrado en las últimas partidas.
6. El rendimiento global se clasifica como alto, pero se debe vigilar la irregularidad entre partidas.


## 13. Predicción por partida

Además de la predicción global, el sistema genera predicciones para cada partida.

Esto permite observar qué partidas fueron positivas, negativas o neutras, y qué estilo se detectó en cada una.

In [16]:
matches_df = pd.DataFrame(analysis["matches"])

matches_df[
    [
        "match_number",
        "date",
        "map",
        "agent",
        "result",
        "acs",
        "kills",
        "deaths",
        "assists",
        "performance_prediction",
        "style_prediction",
        "trend_signal",
    ]
]

,match_number,date,map,agent,result,acs,kills,deaths,assists,performance_prediction,style_prediction,trend_signal
0,1,"30/5/26, 2:12",Ascent,Brimstone,Win,336.0,25,14,5,Destacado,Alto impacto,Positiva
1,2,"30/5/26, 1:36",Fracture,Gekko,Win,290.0,21,12,4,Destacado,Alto impacto,Positiva
2,3,"30/5/26, 0:54",Split,Vyse,Loss,280.0,22,16,7,Alto,Alto impacto,Positiva
3,4,"30/5/26, 0:12",Haven,Brimstone,Win,259.0,19,17,16,Alto,Apoyo táctico,Positiva
4,5,"26/5/26, 21:06",Haven,Cypher,Loss,166.0,10,14,3,Bajo,Apoyo táctico,Negativa
5,6,"19/5/26, 0:11",Ascent,Cypher,Win,223.0,18,15,2,Alto,Ofensivo consistente,Neutra
6,7,"18/5/26, 23:37",Pearl,Gekko,Win,247.0,16,7,3,Destacado,Alto impacto,Positiva
7,8,"18/5/26, 23:03",Ascent,Cypher,Win,211.0,15,10,3,Medio,Ofensivo consistente,Neutra
8,9,"18/5/26, 22:27",Split,Cypher,Win,238.0,18,8,5,Alto,Alto impacto,Positiva
9,10,"18/5/26, 21:44",Breeze,Cypher,Win,289.0,26,10,6,Destacado,Alto impacto,Positiva


## 14. Metodología resumida del sistema

El JSON final también guarda un resumen metodológico para explicar el sistema.

In [17]:
methodology = analysis["methodology"]

print("Problema:")
print(methodology["problem"])
print()

print("Entrada:")
print(methodology["input"])
print()

print("Lógica de comparación por rango:")
print(methodology["rank_reference_logic"])

Problema:
Predecir el rendimiento reciente, el estilo de juego y la tendencia competitiva de un jugador de Valorant a partir de sus últimas partidas competitivas.

Entrada:
Riot ID del jugador. El sistema obtiene sus últimas partidas competitivas desde Tracker.gg y calcula métricas por partida.

Lógica de comparación por rango:
El jugador objetivo se compara contra jugadores de referencia filtrados por la media del rango promedio de sus partidas. Esto evita comparar directamente contra jugadores de contextos competitivos muy distintos.


In [18]:
pd.DataFrame(methodology["tasks"])

,task,target,output,model_info
0,Clasificación de rendimiento por partida,performance_level,"[Bajo, Medio, Alto, Destacado]",{'task': 'Clasificación supervisada de nivel d...
1,Clasificación de estilo de juego por partida y...,player_type,"[Alto impacto, Apoyo táctico, Ofensivo consist...","{'task': 'Clasificación de estilo de juego', '..."
2,Estimación de tendencia competitiva reciente,trend_status,"[Riesgo de bajar, Estable, Progreso positivo, ...",{'task': 'Estimación de tendencia competitiva ...
3,Búsqueda de referentes similares,similar_players,Jugadores de referencia con media de lobby par...,{'task': 'Búsqueda de jugadores similares por ...


## 15. Limitaciones

Es importante declarar las limitaciones del sistema:

- La base de referencia puede crecer para mejorar la comparación por rango.
- La evolución temporal se calcula sobre las últimas 20 partidas, no sobre toda la temporada.
- Clutch rate y utility impact no se miden directamente, sino que se dejan como mejoras futuras o aproximaciones.
- El scraping depende de la disponibilidad y estructura de Tracker.gg.

In [19]:
for limitation in methodology["limitations"]:
    print("-", limitation)

- La base de referencia corresponde a una muestra limitada de jugadores.
- La comparación se realiza por estadísticas recientes agregadas y grupo de rango promedio de partida.
- En una versión de mayor escala, la base puede ampliarse con más jugadores y más partidas por rango.
